# 🔬 Architecture & Dataset Agnostic Ablation Sweeps & Pareto Optimization

This notebook provides a **universally configurable, multi-dimensional ablation engine** across all hybrid architectures and datasets in the Unified LinkNER project.

### 📌 Supported Architectures:
1. **`dinasor_dingen`**: SpanNER + Dinasor Hinting $\rightarrow$ DinGenerator Dynamic Reflection Pipeline.
2. **`hybrid_linkner`**: SpanNER + Single-Pass LLM Arbitration + LOF Novelty Scoring.
3. **`gating_nn`**: SpanNER + Single Consolidated Prompt v1 (Inlined Spans & Feature Evidence from Notebook 02).
4. **`agentic_guidelines`**: Pure LLM Self-Reflective Agentic Guidelines Pipeline (Generator, Reflector, Curator).
5. **`spanner_only`**: Baseline Local SpanNER Neural Model.

### 📌 Key Capabilities:
1. **Architecture-Agnostic**: Select any architecture (`dinasor_dingen`, `hybrid_linkner`, `gating_nn`, `agentic_guidelines`, or `spanner_only`).
2. **Dataset-Agnostic**: Choose any dataset split (`balanced_val`, `balanced_test`, `balanced_train`, `400_test`, `400_val`, `200_test`, `600_all`, `relabeled_lsf200`, or a custom directory).
3. **Single-Pass Multi-Metric Feature Precomputation**: One forward pass extracts MCD, PE, Margin, LC uncertainties, and LOF, Mahalanobis, and kNN novelty scores.
4. **Persistent Zero-Cost LLM Decision Caching**: Eliminates duplicate LLM API calls for 100% deterministic reproducibility.
5. **Sequential Multi-Dimensional Optimization Cascade**:
   - **Sweep 1 (Weights)**: Finds optimal `w_novelty` & `w_uncertainty` $\rightarrow$ feeds into Sweep 2.
   - **Sweep 2 (Operators)**: Finds best Gating Operator (`weighted`, `or`, `and`, `prob_or`, `copula`) $\rightarrow$ feeds into Sweep 3.
   - **Sweep 3 (Uncertainty)**: Finds best Uncertainty Estimator (`MCD`, `PE`, `Margin`, `LC`) $\rightarrow$ feeds into Sweep 4.
   - **Sweep 4 (Novelty)**: Finds best Novelty Metric (`LOF`, `Mahalanobis`, `kNN`) $\rightarrow$ feeds into Sweep 5.
   - **Sweep 5 (Threshold & Pareto)**: Sweeps $\tau \in [0.20, 0.60]$ to generate the Cost vs Accuracy Pareto Frontier $\rightarrow$ feeds into Sweep 6.
   - **Sweep 6 (Component Ablation)**: Tests individual architectural component contributions against the optimal baseline.
6. **Structured Audit Logs & 8-Outcome Diagnostics**: Generates complete audit reports and the 8-outcome diagnostic matrix.

In [ ]:
# ==============================================================================
# [CELL 1]: Environment Bootstrap, Unified Path Setup & Imports
# ==============================================================================
%load_ext autoreload
%autoreload 2

import os
import sys
import glob
import time
import json
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from dotenv import load_dotenv

# 1. Bootstrap Unified Project Paths
NOTEBOOK_DIR = Path(os.getcwd()).resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# 2. Load Central Configuration & Environment Variables
from src.common.config import (
    PROJECT_ROOT as CFG_PROJECT_ROOT,
    DATA_DIR as CFG_DATA_DIR,
    MODELS_DIR as CFG_MODELS_DIR,
    PROMPTS_DIR as CFG_PROMPTS_DIR,
    OUTPUT_DIR as CFG_OUTPUT_DIR,
    DATA_SPLITS
)

load_dotenv(PROJECT_ROOT / ".env", override=True)

device = torch.device("mps") if torch.backends.mps.is_available() else (torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu"))

print("✅ Unified Environment Bootstrapped Successfully.")
print(f"  • Project Root     : {PROJECT_ROOT}")
print(f"  • Data Directory   : {CFG_DATA_DIR}")
print(f"  • Models Directory : {CFG_MODELS_DIR}")
print(f"  • Output Directory : {CFG_OUTPUT_DIR}")
print(f"  • Compute Device   : {device}")

In [ ]:
# ==============================================================================
# [CELL 2]: Architecture & Dataset Selection Hub (Universal Configurator)
# ==============================================================================

# ------------------------------------------------------------------------------
# 1. CHOOSE ARCHITECTURE UNDER TEST
# Supported: 'dinasor_dingen', 'hybrid_linkner', 'gating_nn', 'agentic_guidelines', 'spanner_only'
# ------------------------------------------------------------------------------
CHOSEN_ARCHITECTURE = os.getenv("ABLATION_ARCHITECTURE", "dinasor_dingen")

# ------------------------------------------------------------------------------
# 2. CHOOSE DATASET UNDER TEST
# Supported registry keys: 'balanced_val', 'balanced_test', 'balanced_train', 
# '400_test', '400_val', '400_train', '200_test', '200_train', '600_all', 
# 'relabeled_lsf200', or specify a custom directory path string
# ------------------------------------------------------------------------------
CHOSEN_DATASET = os.getenv("ABLATION_DATASET", "balanced_val")

# Dataset Registry Definitions
DATASET_REGISTRY = {
    "balanced_val": {
        "path": CFG_DATA_DIR / "new/400Abstracts/no_disease/balanced/val",
        "name": "balanced_val_set",
        "description": "Balanced Validation Split (80 Abstracts, No Disease)"
    },
    "balanced_test": {
        "path": CFG_DATA_DIR / "new/400Abstracts/no_disease/balanced/test",
        "name": "balanced_test_set",
        "description": "Balanced Test Split (80 Abstracts, No Disease)"
    },
    "balanced_train": {
        "path": CFG_DATA_DIR / "new/400Abstracts/no_disease/balanced/train",
        "name": "balanced_train_set",
        "description": "Balanced Training Split (240 Abstracts, No Disease)"
    },
    "400_test": {
        "path": CFG_DATA_DIR / "new/400Abstracts/no_disease/test",
        "name": "400_test_set",
        "description": "Standard 400-Split Test Folder"
    },
    "400_val": {
        "path": CFG_DATA_DIR / "new/400Abstracts/no_disease/val",
        "name": "400_val_set",
        "description": "Standard 400-Split Validation Folder"
    },
    "400_train": {
        "path": CFG_DATA_DIR / "new/400Abstracts/no_disease/train",
        "name": "400_train_set",
        "description": "Standard 400-Split Training Folder"
    },
    "200_test": {
        "path": CFG_DATA_DIR / "new/200Abstracts/test",
        "name": "200_test_set",
        "description": "Standard 200-Split Test Set (200 Abstracts)"
    },
    "200_train": {
        "path": CFG_DATA_DIR / "new/200Abstracts/train",
        "name": "200_train_set",
        "description": "Standard 200-Split Training Set"
    },
    "600_all": {
        "path": CFG_DATA_DIR / "600Abstracts",
        "name": "600_all_abstracts",
        "description": "Complete 600 Abstracts Dataset"
    },
    "relabeled_lsf200": {
        "path": CFG_DATA_DIR / "Relabaled_LSF200_NoDisease",
        "name": "relabeled_lsf200",
        "description": "Relabeled LSF200 (No Disease, 10 Canonical Classes)"
    }
}

# Resolve Dataset Path & Clean Identifier Name
if CHOSEN_DATASET in DATASET_REGISTRY:
    info = DATASET_REGISTRY[CHOSEN_DATASET]
    DATA_DIR = str(info["path"])
    DATASET_NAME = info["name"]
    DATASET_DESC = info["description"]
else:
    DATA_DIR = str(Path(CHOSEN_DATASET).resolve())
    DATASET_NAME = Path(DATA_DIR).name
    DATASET_DESC = f"Custom Dataset Path: {DATA_DIR}"

# Fallback Resolution if path doesn't exist
if not os.path.exists(DATA_DIR):
    alt_candidates = [
        str(CFG_DATA_DIR / "new/400Abstracts/no_disease/balanced/val"),
        str(CFG_DATA_DIR / "new/200Abstracts/test"),
        str(CFG_DATA_DIR / "Relabaled_test_folder_noDisease"),
        str(CFG_DATA_DIR / "600Abstracts")
    ]
    DATA_DIR = next((c for c in alt_candidates if os.path.exists(c)), DATA_DIR)
    DATASET_NAME = Path(DATA_DIR).name
    DATASET_DESC = f"Auto-Resolved Fallback: {DATA_DIR}"

# ------------------------------------------------------------------------------
# 3. LLM PROVIDER & RUNTIME HYPERPARAMETERS
# ------------------------------------------------------------------------------
SEED = int(os.getenv("RANDOM_SEED", "42"))
torch.manual_seed(SEED)
np.random.seed(SEED)

LLM_PROVIDER = os.getenv("LLM_PROVIDER", "9router")
MODEL_NAME = os.getenv("LLM_MODEL", os.getenv("OPENROUTER_MODEL", "gemini-cli"))
BACKEND = os.getenv("LLM_BACKEND", "api")
BATCH_SIZE = int(os.getenv("BATCH_SIZE", "5"))  # Concurrency: 1 worker thread per abstract

# Ablation Output Root Directory
OUTPUT_FOLDER = str(CFG_OUTPUT_DIR / "ablation_reports" / CHOSEN_ARCHITECTURE)
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# Initial Baseline Sweep Hyperparameters
CURRENT_GATING_MODE = "weighted"
CURRENT_W_NOVELTY = 0.50
CURRENT_W_UNCERTAINTY = 0.50
CURRENT_UNCERTAINTY_METHOD = "mcd"
CURRENT_NOVELTY_METRIC = "lof"
CURRENT_THRESHOLD = 0.38
CURRENT_MARGIN_THRESHOLD = 0.50

print("=" * 92)
print("  📋 EXPERIMENT CONFIGURATION: ARCHITECTURE & DATASET SELECTION")
print("=" * 92)
print(f"  • Chosen Architecture : {CHOSEN_ARCHITECTURE.upper()}")
print(f"  • Chosen Dataset      : {DATASET_NAME} ({DATASET_DESC})")
print(f"  • Dataset Path        : {DATA_DIR}")
print(f"  • Active LLM Model    : {MODEL_NAME} (Provider: {LLM_PROVIDER.upper()}, Backend: {BACKEND})")
print(f"  • Worker Concurrency  : {BATCH_SIZE} threads per batch")
print(f"  • Output Directory    : {OUTPUT_FOLDER}")
print("=" * 92)

In [ ]:
# ==============================================================================
# [CELL 3]: Pluggable Architecture Pipeline & Model Construction
# ==============================================================================
from transformers import AutoTokenizer
from src.architectures.hybrid_linkner.spanner_model import SpanNERModel
from src.architectures.hybrid_linkner.novelty_detector import build_lof_novelty_detector
from src.common.label_mapping import CANONICAL_10_ID2LABEL, canonicalize_label

# 1. Model & Tokenizer Paths
SPANNER_ENCODER_PATH = str(CFG_MODELS_DIR / "NER_Model/trained_NER_model")
SPANNER_CHECKPOINT_PATH = str(CFG_MODELS_DIR / "SpanNER_LSF/best_spanner_160train.pt")

ID2LABEL = CANONICAL_10_ID2LABEL
tokenizer = AutoTokenizer.from_pretrained(SPANNER_ENCODER_PATH)
spanner_model = SpanNERModel(encoder_path=SPANNER_ENCODER_PATH, num_classes=len(ID2LABEL), max_span_width=6)
spanner_model.load_state_dict(torch.load(SPANNER_CHECKPOINT_PATH, map_location=device))
spanner_model.to(device).eval()

novelty_detector = build_lof_novelty_detector(k=10, alpha=0.5)

# 2. Build Pipeline according to CHOSEN_ARCHITECTURE
pipeline = None

if CHOSEN_ARCHITECTURE == "dinasor_dingen":
    from src.architectures.dinasor_dingen import Dinasor, DinGenerator, SpanNER_DinGenPipeline
    dinasor_agent = Dinasor(model_name=MODEL_NAME, backend=BACKEND)
    dingenerator_agent = DinGenerator(model_name=MODEL_NAME, backend=BACKEND)
    pipeline = SpanNER_DinGenPipeline(
        spanner_model=spanner_model,
        tokenizer=tokenizer,
        novelty_scorer=novelty_detector,
        dinasor_agent=dinasor_agent,
        dingenerator_agent=dingenerator_agent,
        device=device,
        id2label=ID2LABEL,
        use_new_labels=True,
        output_folder=OUTPUT_FOLDER
    )

elif CHOSEN_ARCHITECTURE == "hybrid_linkner":
    from src.architectures.hybrid_linkner.pipeline import LinkNERPipeline, get_llm_provider
    llm_prov = get_llm_provider()
    pipeline = LinkNERPipeline(
        spanner_model_path=SPANNER_ENCODER_PATH,
        spanner_checkpoint_path=SPANNER_CHECKPOINT_PATH,
        llm_provider=llm_prov,
        novelty_scorer=novelty_detector,
        use_new_labels=True
    )
    pipeline.spanner_model = spanner_model
    pipeline.tokenizer = tokenizer
    pipeline.device = device
    pipeline.id2label = ID2LABEL
    pipeline.num_classes = len(ID2LABEL)

elif CHOSEN_ARCHITECTURE == "gating_nn":
    from src.architectures.gating_nn import GatingNNPipeline
    from src.architectures.hybrid_linkner.pipeline import get_llm_provider
    llm_prov = get_llm_provider()
    pipeline = GatingNNPipeline(
        spanner_model=spanner_model,
        tokenizer=tokenizer,
        novelty_scorer=novelty_detector,
        llm_provider=llm_prov,
        prompt_version="v1",
        device=device,
        id2label=ID2LABEL,
        use_new_labels=True,
        output_folder=OUTPUT_FOLDER
    )

elif CHOSEN_ARCHITECTURE == "agentic_guidelines":
    from src.architectures.agentic_guidelines import AgenticNERPipeline
    pipeline = AgenticNERPipeline(
        model_name=MODEL_NAME,
        backend=BACKEND,
        use_new_labels=True
    )
    pipeline.spanner_model = spanner_model
    pipeline.tokenizer = tokenizer
    pipeline.device = device
    pipeline.id2label = ID2LABEL
    pipeline.num_classes = len(ID2LABEL)
    pipeline.novelty_scorer = novelty_detector

else:  # 'spanner_only' or baseline
    from src.architectures.dinasor_dingen import SpanNER_DinGenPipeline
    pipeline = SpanNER_DinGenPipeline(
        spanner_model=spanner_model,
        tokenizer=tokenizer,
        novelty_scorer=novelty_detector,
        dinasor_agent=None,
        dingenerator_agent=None,
        device=device,
        id2label=ID2LABEL,
        use_new_labels=True,
        output_folder=OUTPUT_FOLDER
    )

print(f"✅ Initialized Architecture Pipeline for '{CHOSEN_ARCHITECTURE.upper()}'.")
print(f"  • Pipeline Class  : {pipeline.__class__.__name__}")
print(f"  • Target Classes  : {len(ID2LABEL)} Canonical LSF Classes")

In [ ]:
# ==============================================================================
# [CELL 4]: Ingest Selected Dataset & High-Speed Feature Precomputation
# ==============================================================================
from src.data.dataset_converter import parse_ann_file
from src.architectures.ablation_engine.sweep_engine import (
    PrecomputedFeatureCache,
    PersistentLLMDecisionCache,
    AblationEngine
)

# 1. Discover and Ingest Dataset Files (.ann + .txt or .bio)
all_ann = sorted(glob.glob(os.path.join(DATA_DIR, "*.ann")))
all_texts, all_doc_ids, all_doc_gts = [], [], []

if all_ann:
    for ann_path in all_ann:
        gt_ents = parse_ann_file(ann_path)
        txt_path = ann_path.replace(".ann", ".txt")
        if os.path.exists(txt_path):
            with open(txt_path, "r", encoding="utf-8") as f:
                all_texts.append(f.read())
            all_doc_ids.append(os.path.basename(txt_path).replace(".txt", ""))
            all_doc_gts.append(gt_ents)
else:
    # Fallback to direct .txt files
    all_txt = sorted(glob.glob(os.path.join(DATA_DIR, "*.txt")))
    for txt_path in all_txt:
        with open(txt_path, "r", encoding="utf-8") as f:
            all_texts.append(f.read())
        all_doc_ids.append(os.path.basename(txt_path).replace(".txt", ""))
        all_doc_gts.append([])

print(f"📂 Loaded {len(all_texts)} evaluation abstracts from: {DATA_DIR}")

# 2. Precompute Multi-Metric Feature Cache (MCD, PE, Margin, LC, LOF, Mahalanobis, kNN)
feature_cache = PrecomputedFeatureCache(
    pipeline=pipeline,
    texts=all_texts,
    doc_ids=all_doc_ids,
    doc_gts=all_doc_gts,
    mcd_passes=5
)

# 3. Persistent Zero-Cost LLM Decision Cache
cache_file = str(CFG_OUTPUT_DIR / "ablation_reports" / f".cache_{CHOSEN_ARCHITECTURE}_decisions.json")
llm_cache = PersistentLLMDecisionCache(cache_file=cache_file)

# 4. Initialize Architecture-Agnostic Ablation Engine
ablation_engine = AblationEngine(
    pipeline=pipeline,
    feature_cache=feature_cache,
    llm_cache=llm_cache,
    output_base_dir=OUTPUT_FOLDER,
    dataset_name=DATASET_NAME,
    batch_size=BATCH_SIZE
)

print(f"✅ Ablation Engine Ready.")
print(f"  • Architecture : {CHOSEN_ARCHITECTURE.upper()}")
print(f"  • Dataset Name : {DATASET_NAME}")
print(f"  • Output Path  : {OUTPUT_FOLDER}/<Test Name>/{DATASET_NAME}/<run_name>/")

In [ ]:
# ==============================================================================
# [CELL 5]: Sweep 1 — Weight Combination Sensitivity (w_novelty vs. w_uncertainty)
# Grid searches w_nov in [0.0, 1.0] and extracts the best weights!
# ==============================================================================
weight_results = ablation_engine.run_weight_combination_sweep(
    step_size=0.10,
    default_threshold=CURRENT_THRESHOLD,
    gating_mode=CURRENT_GATING_MODE,
    uncertainty_method=CURRENT_UNCERTAINTY_METHOD,
    novelty_metric=CURRENT_NOVELTY_METRIC,
    test_name="Weight Combination Sensitivity",
    dataset_name=DATASET_NAME,
    log_run_files=True
)

# Display Interactive DataFrame Table
df_weights = pd.DataFrame([
    {
        "w_Novelty": r["config"]["w_novelty"],
        "w_Uncertainty": r["config"]["w_uncertainty"],
        "Macro F1 (%)": round(r["metrics"]["macro_f1"] * 100, 2),
        "Micro F1 (%)": round(r["metrics"]["micro_f1"] * 100, 2),
        "Micro Precision (%)": round(r["metrics"]["micro_precision"] * 100, 2),
        "Micro Recall (%)": round(r["metrics"]["micro_recall"] * 100, 2),
        "Accuracy (%)": round(r["metrics"]["span_accuracy"] * 100, 2),
        "Escalation Rate (%)": round(r["efficiency"]["escalation_rate_pct"], 2),
        "Local Saved (%)": round(r["efficiency"]["local_saved_pct"], 2),
        "Run Log Directory": r.get("run_dir", "")
    }
    for r in weight_results
])

print("\n--- 📊 Weight Sensitivity Grid Search Table ---")
display(df_weights)

# Extract Best Weight Combination to feed forward into subsequent sweeps
best_weight_res = max(weight_results, key=lambda x: x["metrics"]["macro_f1"])
CURRENT_W_NOVELTY = float(best_weight_res["config"]["w_novelty"])
CURRENT_W_UNCERTAINTY = float(best_weight_res["config"]["w_uncertainty"])

print("\n" + "=" * 88)
print("  🏆 [STAGE 1 COMPLETE] BEST WEIGHT COMBINATION SELECTED:")
print(f"  • w_Novelty     : {CURRENT_W_NOVELTY:.2f}")
print(f"  • w_Uncertainty : {CURRENT_W_UNCERTAINTY:.2f}")
print(f"  • Best Macro F1 : {best_weight_res['metrics']['macro_f1']*100:.2f}% (Micro F1: {best_weight_res['metrics']['micro_f1']*100:.2f}%)")
print(f"  • Run Log       : {best_weight_res.get('run_dir', 'N/A')}")
print("=" * 88)

In [ ]:
# ==============================================================================
# [CELL 6]: Sweep 2 — Gating Fusion Operators (weighted, or, and, prob_or, copula)
# Uses optimal weights (w_nov, w_unc) from Cell 5 as base configuration!
# ==============================================================================
operator_results = ablation_engine.run_gating_operator_sweep(
    default_threshold=CURRENT_THRESHOLD,
    w_novelty=CURRENT_W_NOVELTY,
    w_uncertainty=CURRENT_W_UNCERTAINTY,
    uncertainty_method=CURRENT_UNCERTAINTY_METHOD,
    novelty_metric=CURRENT_NOVELTY_METRIC,
    test_name="Gating Fusion Operators",
    dataset_name=DATASET_NAME,
    log_run_files=True
)

df_operators = pd.DataFrame([
    {
        "Operator": r["config"]["gating_mode"].upper(),
        "Macro F1 (%)": round(r["metrics"]["macro_f1"] * 100, 2),
        "Micro F1 (%)": round(r["metrics"]["micro_f1"] * 100, 2),
        "Micro Precision (%)": round(r["metrics"]["micro_precision"] * 100, 2),
        "Micro Recall (%)": round(r["metrics"]["micro_recall"] * 100, 2),
        "Accuracy (%)": round(r["metrics"]["span_accuracy"] * 100, 2),
        "Escalation Rate (%)": round(r["efficiency"]["escalation_rate_pct"], 2),
        "Run Log Directory": r.get("run_dir", "")
    }
    for r in operator_results
])

print("\n--- 📊 Gating Fusion Operators Comparison ---")
display(df_operators)

# Extract Best Gating Operator to feed forward into subsequent sweeps
best_op_res = max(operator_results, key=lambda x: x["metrics"]["macro_f1"])
CURRENT_GATING_MODE = str(best_op_res["config"]["gating_mode"])

print("\n" + "=" * 88)
print("  🏆 [STAGE 2 COMPLETE] BEST GATING OPERATOR SELECTED:")
print(f"  • Gating Operator : {CURRENT_GATING_MODE.upper()}")
print(f"  • Base Weights    : w_nov={CURRENT_W_NOVELTY:.2f}, w_unc={CURRENT_W_UNCERTAINTY:.2f}")
print(f"  • Best Macro F1   : {best_op_res['metrics']['macro_f1']*100:.2f}% (Micro F1: {best_op_res['metrics']['micro_f1']*100:.2f}%)")
print(f"  • Run Log         : {best_op_res.get('run_dir', 'N/A')}")
print("=" * 88)

In [ ]:
# ==============================================================================
# [CELL 7]: Sweep 3 — Uncertainty Estimators (MCD, PE, Margin, LC)
# Uses optimal weights (Cell 5) & operator (Cell 6) as base configuration!
# ==============================================================================
unc_results = ablation_engine.run_uncertainty_methods_sweep(
    default_threshold=CURRENT_THRESHOLD,
    gating_mode=CURRENT_GATING_MODE,
    w_novelty=CURRENT_W_NOVELTY,
    w_uncertainty=CURRENT_W_UNCERTAINTY,
    novelty_metric=CURRENT_NOVELTY_METRIC,
    test_name="Uncertainty Estimators",
    dataset_name=DATASET_NAME,
    log_run_files=True
)

df_unc = pd.DataFrame([
    {
        "Uncertainty Method": r["config"]["uncertainty_method"].upper(),
        "Macro F1 (%)": round(r["metrics"]["macro_f1"] * 100, 2),
        "Micro F1 (%)": round(r["metrics"]["micro_f1"] * 100, 2),
        "Micro Precision (%)": round(r["metrics"]["micro_precision"] * 100, 2),
        "Micro Recall (%)": round(r["metrics"]["micro_recall"] * 100, 2),
        "Accuracy (%)": round(r["metrics"]["span_accuracy"] * 100, 2),
        "Run Log Directory": r.get("run_dir", "")
    }
    for r in unc_results
])

print("\n--- 📊 Uncertainty Estimators Comparison ---")
display(df_unc)

# Extract Best Uncertainty Method to feed forward into subsequent sweeps
best_unc_res = max(unc_results, key=lambda x: x["metrics"]["macro_f1"])
CURRENT_UNCERTAINTY_METHOD = str(best_unc_res["config"]["uncertainty_method"])

print("\n" + "=" * 88)
print("  🏆 [STAGE 3 COMPLETE] BEST UNCERTAINTY ESTIMATOR SELECTED:")
print(f"  • Uncertainty Method : {CURRENT_UNCERTAINTY_METHOD.upper()}")
print(f"  • Base Configuration : Operator={CURRENT_GATING_MODE.upper()}, w_nov={CURRENT_W_NOVELTY:.2f}, w_unc={CURRENT_W_UNCERTAINTY:.2f}")
print(f"  • Best Macro F1      : {best_unc_res['metrics']['macro_f1']*100:.2f}% (Micro F1: {best_unc_res['metrics']['micro_f1']*100:.2f}%)")
print(f"  • Run Log            : {best_unc_res.get('run_dir', 'N/A')}")
print("=" * 88)

In [ ]:
# ==============================================================================
# [CELL 8]: Sweep 4 — Novelty Density Estimators (LOF, Mahalanobis, kNN)
# Uses optimal weights, operator, and uncertainty method as base configuration!
# ==============================================================================
nov_results = ablation_engine.run_novelty_metrics_sweep(
    default_threshold=CURRENT_THRESHOLD,
    gating_mode=CURRENT_GATING_MODE,
    w_novelty=CURRENT_W_NOVELTY,
    w_uncertainty=CURRENT_W_UNCERTAINTY,
    uncertainty_method=CURRENT_UNCERTAINTY_METHOD,
    test_name="Novelty Density Estimators",
    dataset_name=DATASET_NAME,
    log_run_files=True
)

df_nov = pd.DataFrame([
    {
        "Novelty Metric": r["config"]["novelty_metric"].upper(),
        "Macro F1 (%)": round(r["metrics"]["macro_f1"] * 100, 2),
        "Micro F1 (%)": round(r["metrics"]["micro_f1"] * 100, 2),
        "Micro Precision (%)": round(r["metrics"]["micro_precision"] * 100, 2),
        "Accuracy (%)": round(r["metrics"]["span_accuracy"] * 100, 2),
        "Run Log Directory": r.get("run_dir", "")
    }
    for r in nov_results
])

print("\n--- 📊 Novelty Density Estimators Comparison ---")
display(df_nov)

# Extract Best Novelty Metric to feed forward into subsequent sweeps
best_nov_res = max(nov_results, key=lambda x: x["metrics"]["macro_f1"])
CURRENT_NOVELTY_METRIC = str(best_nov_res["config"]["novelty_metric"])

print("\n" + "=" * 88)
print("  🏆 [STAGE 4 COMPLETE] CUMULATIVE BEST BASE CONFIGURATION (AFTER SWEEPS 1-4):")
print(f"  • Gating Operator    : {CURRENT_GATING_MODE.upper()}")
print(f"  • Weight Combination : w_nov={CURRENT_W_NOVELTY:.2f}, w_unc={CURRENT_W_UNCERTAINTY:.2f}")
print(f"  • Uncertainty Method : {CURRENT_UNCERTAINTY_METHOD.upper()}")
print(f"  • Novelty Metric     : {CURRENT_NOVELTY_METRIC.upper()}")
print(f"  • Best Macro F1      : {best_nov_res['metrics']['macro_f1']*100:.2f}% (Micro F1: {best_nov_res['metrics']['micro_f1']*100:.2f}%)")
print(f"  • Run Log            : {best_nov_res.get('run_dir', 'N/A')}")
print("=" * 88)

In [ ]:
# ==============================================================================
# [CELL 9]: Sweep 5 — Threshold Sensitivity & Pareto Frontier (Cost vs. Accuracy)
# Sweeps tau in [0.20, 0.60] on top of the cumulative best configuration (Cells 5-8)!
# ==============================================================================
thresh_results = ablation_engine.run_threshold_pareto_sweep(
    step_size=0.05,
    gating_mode=CURRENT_GATING_MODE,
    w_novelty=CURRENT_W_NOVELTY,
    w_uncertainty=CURRENT_W_UNCERTAINTY,
    uncertainty_method=CURRENT_UNCERTAINTY_METHOD,
    novelty_metric=CURRENT_NOVELTY_METRIC,
    test_name="Threshold Sensitivity",
    dataset_name=DATASET_NAME,
    log_run_files=True
)

df_pareto = pd.DataFrame([
    {
        "Threshold (Tau)": r["config"]["threshold"],
        "Macro F1 (%)": round(r["metrics"]["macro_f1"] * 100, 2),
        "Micro F1 (%)": round(r["metrics"]["micro_f1"] * 100, 2),
        "Micro Precision (%)": round(r["metrics"]["micro_precision"] * 100, 2),
        "Micro Recall (%)": round(r["metrics"]["micro_recall"] * 100, 2),
        "Accuracy (%)": round(r["metrics"]["span_accuracy"] * 100, 2),
        "Escalation Rate (%)": round(r["efficiency"]["escalation_rate_pct"], 2),
        "Local Computation Saved (%)": round(r["efficiency"]["local_saved_pct"], 2),
        "Run Log Directory": r.get("run_dir", "")
    }
    for r in thresh_results
])

print("\n--- 📊 Pareto Frontier Table (Cost / Escalation vs. F1 & Accuracy) ---")
display(df_pareto)

# Extract Best Threshold (Tau) to feed forward into component ablation
best_thresh_res = max(thresh_results, key=lambda x: x["metrics"]["macro_f1"])
CURRENT_THRESHOLD = float(best_thresh_res["config"]["threshold"])

print("\n" + "=" * 88)
print("  🏆 [STAGE 5 COMPLETE] OPTIMAL UNRELIABILITY THRESHOLD (TAU) SELECTED:")
print(f"  • Optimal Tau (Threshold) : {CURRENT_THRESHOLD:.2f}")
print(f"  • Macro F1-Score          : {best_thresh_res['metrics']['macro_f1']*100:.2f}% (Micro F1: {best_thresh_res['metrics']['micro_f1']*100:.2f}%)")
print(f"  • Escalation Rate         : {best_thresh_res['efficiency']['escalation_rate_pct']:.1f}%")
print(f"  • Run Log                 : {best_thresh_res.get('run_dir', 'N/A')}")
print("=" * 88)

In [ ]:
# ==============================================================================
# [CELL 10]: Sweep 6 — Architectural Feature Component Ablation Matrix
# Tests feature ablation switches starting from the optimal baseline (Cells 5-9)!
# ==============================================================================
comp_results = ablation_engine.run_component_ablation_matrix(
    gating_mode=CURRENT_GATING_MODE,
    w_novelty=CURRENT_W_NOVELTY,
    w_uncertainty=CURRENT_W_UNCERTAINTY,
    uncertainty_method=CURRENT_UNCERTAINTY_METHOD,
    novelty_metric=CURRENT_NOVELTY_METRIC,
    unreliability_threshold=CURRENT_THRESHOLD,
    margin_threshold=CURRENT_MARGIN_THRESHOLD,
    test_name="Architectural Component Ablation",
    dataset_name=DATASET_NAME,
    log_run_files=True
)

df_comp = pd.DataFrame([
    {
        "Component Configuration": r["component_name"],
        "Macro F1 (%)": round(r["metrics"]["macro_f1"] * 100, 2),
        "Micro F1 (%)": round(r["metrics"]["micro_f1"] * 100, 2),
        "Micro Precision (%)": round(r["metrics"]["micro_precision"] * 100, 2),
        "Accuracy (%)": round(r["metrics"]["span_accuracy"] * 100, 2),
        "Delta Macro F1 (%)": round(r["metrics"]["delta_macro_f1"] * 100, 2),
        "Run Log Directory": r.get("run_dir", "")
    }
    for r in comp_results
])

print("\n--- 📊 Architectural Feature Component Ablation Matrix ---")
display(df_comp)

In [ ]:
# ==============================================================================
# [CELL 11]: Master Report Export & 8-Outcome Escalation Audit Diagnostics
# Exports master report files & displays 8-outcome diagnostic breakdown!
# ==============================================================================
from src.common.audit_logger import (
    compute_audit_breakdown,
    format_audit_breakdown_table,
    format_audit_summary_box
)

# 1. Export Master Ablation Report & Sweep CSVs
report_txt, report_json = ablation_engine.export_master_ablation_report(
    weight_results=weight_results,
    operator_results=operator_results,
    unc_results=unc_results,
    nov_results=nov_results,
    thresh_results=thresh_results,
    comp_results=comp_results,
    dataset_name=DATASET_NAME
)

print("\n" + "=" * 90)
print("  ✅ ABLATION EXPERIMENTS & MASTER REPORT EXPORT COMPLETED!")
print(f"  • Architecture        : {CHOSEN_ARCHITECTURE.upper()}")
print(f"  • Dataset             : {DATASET_NAME}")
print(f"  • Master Summary TXT  : {report_txt}")
print(f"  • Master Metrics JSON : {report_json}")
print("=" * 90)

with open(report_txt, "r", encoding="utf-8") as f:
    print("\n" + f.read())

# 2. Compute 8-Outcome Generative LLM Escalation Audit Diagnostics for Global Best Run
all_runs = weight_results + operator_results + unc_results + nov_results + thresh_results + comp_results
best_res = max(all_runs, key=lambda x: x["metrics"]["macro_f1"])

escalated_items = [
    {
        "spanner_label": r["spanner_label"],
        "final_label": r["hybrid_label"],
        "ground_truth_label": r["ground_truth"],
        "llm_label": r["hybrid_label"]
    }
    for r in best_res.get("audit_records", [])
    if r.get("escalated", False)
]

breakdown = compute_audit_breakdown(escalated_items)
best_unc_method_display = best_res["config"].get("uncertainty_method", "mcd").upper()

print("\n" + format_audit_summary_box(breakdown, method=best_unc_method_display))
print("\n" + "=" * 96)
print(f"              DETAILED GENERATIVE LLM ESCALATION AUDIT BREAKDOWN ({CHOSEN_ARCHITECTURE.upper()})")
print("=" * 96)
print(format_audit_breakdown_table(breakdown))
print(f"\n📁 Optimal Run Directory: {best_res.get('run_dir', 'N/A')}")